In [1]:
from astropy.time import Time
from sora.star import Star


SORA version: 0.3.3


In [2]:
# IDs
body_id = 'Silesia'
star_id = '4051506284553476736'
obs_id = 'W63'

# Epochs
epoch_jpl = {
  'start': '2019-06-27',
  'stop': '2019-06-29',
  'step': '1m',
  'number': 1000
}
epoch_mpc = {
  'start': '2019-06-27',
  'stop': '2019-06-29',
  'step': '1min',
  'number': 1000
}
epoch_occ = Time("2019-06-28 03:48:10.040")

In [3]:
# Instantiate star
star = Star(code=star_id, epoch=epoch_occ, nomad=False)

1 GaiaDR3 (LIneA TAP: gaia_dr3.source) star found band={'G': 13.397762298583984}
star coordinate at J2016.0: RA=18h20m24.29259s +/- 0.019742164760828018 deg, DEC=-27d51m06.4713s +/- 0.016225365921854973 deg


In [4]:
import numpy as np
from astropy import units as u
from sora.star.utils import spatial_motion

# --------------------------------------------------
# 1. Define parameter orders
# --------------------------------------------------

gaia_order = ['ra', 'dec', 'pmra', 'pmdec', 'parallax', 'rad_vel']
sora_order = ['ra', 'dec', 'parallax', 'pmra', 'pmdec', 'rad_vel']

# --------------------------------------------------
# 2. Reconstruct covariance in Gaia parameter order
# --------------------------------------------------

cov_gaia = np.zeros((6, 6))

for i, key in enumerate(gaia_order):
    cov_gaia[i, i] = star.errors[key].value**2

# Gaia correlations
index_gaia = {key: i for i, key in enumerate(gaia_order)}

for (key1, key2), corr_column in star.catalogue.correlations.items():
    i = index_gaia[key1]
    j = index_gaia[key2]

    error1 = star.errors[key1].value
    error2 = star.errors[key2].value
    correlation = star.meta_catalogue[corr_column]

    cov_gaia[i, j] = correlation * error1 * error2
    cov_gaia[j, i] = cov_gaia[i, j]

# --------------------------------------------------
# 3. Reorder covariance to SORA parameter order
# --------------------------------------------------

permutation = [gaia_order.index(key) for key in sora_order]

cov_correct = cov_gaia[np.ix_(permutation, permutation)]

# --------------------------------------------------
# 4. Compare with original SORA covariance
# --------------------------------------------------

print("Original SORA covariance:")
print(star.cov)

print("\nReconstructed covariance (SORA order):")
print(cov_correct)

print("\nDifference:")
print(star.cov - cov_correct)

# --------------------------------------------------
# 5. Propagate both covariance matrices
# --------------------------------------------------

def propagate_covariance(covariance):
    _, _, _, _, C = spatial_motion(
        star.ra,
        star.dec,
        star.pmra,
        star.pmdec,
        star.parallax,
        star.rad_vel,
        dt=(epoch_occ - star.epoch).jd,
        cov_matrix=covariance
    )
    return C

Original SORA covariance:
[[ 3.89753069e-04  2.67843243e-05 -8.11065474e-05 -4.82441355e-05
   1.83483371e-05  9.92971652e-12]
 [ 2.67843243e-05  2.63262499e-04 -9.77969758e-05  2.01727962e-05
  -1.00032739e-04  1.19730931e-11]
 [-8.11065474e-05 -9.77969758e-05  6.38914766e-04  7.09598098e-05
   7.62017298e-05 -7.82210896e-11]
 [-4.82441355e-05  2.01727962e-05  7.09598098e-05  3.26566275e-04
   5.98927545e-05 -8.68747121e-12]
 [ 1.83483371e-05 -1.00032739e-04  7.62017298e-05  5.98927545e-05
   4.80905694e-04 -9.32922926e-12]
 [ 9.92971652e-12  1.19730931e-11 -7.82210896e-11 -8.68747121e-12
  -9.32922926e-12  9.68062303e-18]]

Reconstructed covariance (SORA order):
[[ 3.89753069e-04  2.67843240e-05 -8.11065416e-05 -4.82441373e-05
   1.83483365e-05  0.00000000e+00]
 [ 2.67843240e-05  2.63262499e-04 -9.77969696e-05  2.01727950e-05
  -1.00032743e-04  0.00000000e+00]
 [-8.11065416e-05 -9.77969696e-05  4.80905694e-04  7.09598066e-05
   7.62017312e-05  0.00000000e+00]
 [-4.82441373e-05  2.017

In [5]:
import numpy as np
from sora.star.utils import spatial_motion

# --------------------------------------------------
# 1. Construct corrected covariance
# --------------------------------------------------

# Preserve the original radial velocity terms.
# Replace only the astrometric 5x5 block.
cov_fixed = star.cov.copy()
cov_fixed[:5, :5] = cov_correct[:5, :5]

# --------------------------------------------------
# 2. Propagate both covariance matrices
# --------------------------------------------------

dt = (epoch_occ - star.epoch).jd

def propagate_covariance(covariance):
    coord, errors, C = spatial_motion(
        star.ra,
        star.dec,
        star.pmra,
        star.pmdec,
        star.parallax,
        star.rad_vel,
        dt=dt,
        cov_matrix=covariance
    )
    return C

C_original = propagate_covariance(star.cov)
C_fixed = propagate_covariance(cov_fixed)

# --------------------------------------------------
# 3. Extract propagated position covariance
# --------------------------------------------------

pos_original = C_original[:2, :2]
pos_fixed = C_fixed[:2, :2]

print("Original propagated position covariance:")
print(pos_original)

print("\nCorrected propagated position covariance:")
print(pos_fixed)

print("\nDifference:")
print(pos_fixed - pos_original)

# --------------------------------------------------
# 4. Compare positional uncertainties
# --------------------------------------------------

sigma_original = np.sqrt(np.diag(pos_original))
sigma_fixed = np.sqrt(np.diag(pos_fixed))

print("\nPositional uncertainties (mas):")
print(f"Original: RA* = {sigma_original[0]:.8f}, "
      f"DEC = {sigma_original[1]:.8f}")
print(f"Corrected: RA* = {sigma_fixed[0]:.8f}, "
      f"DEC = {sigma_fixed[1]:.8f}")

print("\nRelative change (%):")
print(100 * (sigma_fixed - sigma_original) / sigma_original)

# --------------------------------------------------
# 5. Compare correlation between RA* and DEC
# --------------------------------------------------

rho_original = (
    pos_original[0, 1] /
    np.sqrt(pos_original[0, 0] * pos_original[1, 1])
)

rho_fixed = (
    pos_fixed[0, 1] /
    np.sqrt(pos_fixed[0, 0] * pos_fixed[1, 1])
)

print("\nPosition correlation:")
print(f"Original: {rho_original:.8f}")
print(f"Corrected: {rho_fixed:.8f}")


Original propagated position covariance:
[[0.00402427 0.00088939]
 [0.00088939 0.00541333]]

Corrected propagated position covariance:
[[0.00782236 0.00088939]
 [0.00088939 0.00353659]]

Difference:
[[ 3.79809220e-03 -2.39311640e-10]
 [-2.39311640e-10 -1.87673502e-03]]

Positional uncertainties (mas):
Original: RA* = 0.06343711, DEC = 0.07357533
Corrected: RA* = 0.08844410, DEC = 0.05946927

Relative change (%):
[ 39.42012653 -19.17226942]

Position correlation:
Original: 0.19055430
Corrected: 0.16909578


In [8]:

import numpy as np
from astropy import units as u
from sora.star.utils import spatial_motion

# --------------------------------------------------
# 1. Reconstruct covariance in SORA parameter order
# --------------------------------------------------

order = ['ra', 'dec', 'parallax', 'pmra', 'pmdec', 'rad_vel']
index = {key: i for i, key in enumerate(order)}

cov_full = np.zeros((6, 6))

# Astrometric variances
for key in order[:5]:
    i = index[key]
    cov_full[i, i] = star.errors[key].value**2

# Gaia correlations
for (key1, key2), corr_column in star.catalogue.correlations.items():
    i, j = index[key1], index[key2]

    rho = star.meta_catalogue[corr_column]

    cov_full[i, j] = (
        rho * star.errors[key1].value * star.errors[key2].value
    )
    cov_full[j, i] = cov_full[i, j]

# --------------------------------------------------
# 2. Radial velocity terms (SORA original formula)
# --------------------------------------------------

A = (1 * u.AU).to(u.km).value

cov_full[5, 5] = (
    cov_full[2, 2] *
    (star.rad_vel.value**2 + star.errors['rad_vel'].value**2) / A**2
    + (star.parallax.to(u.rad).value *
       star.errors['rad_vel'].value / A)**2
)

for i in range(5):
    cov_full[i, 5] = (
        cov_full[i, 2] * star.rad_vel.value / A
    )
    cov_full[5, i] = cov_full[i, 5]

# --------------------------------------------------
# 3. Check covariance matrix
# --------------------------------------------------

print("Corrected covariance:")
print(cov_full)

print("\nEigenvalues:")
print(np.linalg.eigvalsh(cov_full))

# --------------------------------------------------
# 4. Propagate both matrices
# --------------------------------------------------

dt = (epoch_occ - star.epoch).jd

def propagate(covariance):
    _, _, C = spatial_motion(
        star.ra,
        star.dec,
        star.pmra,
        star.pmdec,
        star.parallax,
        star.rad_vel,
        dt=dt,
        cov_matrix=covariance
    )
    return C

C_original = propagate(star.cov)
C_corrected = propagate(cov_full)

# --------------------------------------------------
# 5. Compare propagated covariance matrices
# --------------------------------------------------

print("\nOriginal propagated covariance:")
print(C_original)

print("\nCorrected propagated covariance:")
print(C_corrected)

print("\nDifference:")
print(C_corrected - C_original)

# --------------------------------------------------
# 6. Compare positional uncertainties
# --------------------------------------------------

sigma_original = np.sqrt(np.diag(C_original)[:2])
sigma_corrected = np.sqrt(np.diag(C_corrected)[:2])

print("\nPosition uncertainties (mas):")
print(f"Original:  RA*={sigma_original[0]:.8f}, "
      f"DEC={sigma_original[1]:.8f}")
print(f"Corrected: RA*={sigma_corrected[0]:.8f}, "
      f"DEC={sigma_corrected[1]:.8f}")

print("\nRelative changes (%):")
print(100 * (sigma_corrected - sigma_original) / sigma_original)

# --------------------------------------------------
# 7. Check positive semidefiniteness
# --------------------------------------------------

print("\nMinimum eigenvalue:")
print(np.linalg.eigvalsh(C_corrected).min())


Corrected covariance:
[[ 3.89753069e-04  2.67843240e-05 -8.11065416e-05 -4.82441373e-05
   1.83483365e-05  9.92971581e-12]
 [ 2.67843240e-05  2.63262499e-04 -9.77969696e-05  2.01727950e-05
  -1.00032743e-04  1.19730924e-11]
 [-8.11065416e-05 -9.77969696e-05  4.80905694e-04  7.09598066e-05
   7.62017312e-05 -5.88763468e-11]
 [-4.82441373e-05  2.01727950e-05  7.09598066e-05  6.38914766e-04
   5.98927536e-05 -8.68747081e-12]
 [ 1.83483365e-05 -1.00032743e-04  7.62017312e-05  5.98927536e-05
   3.26566275e-04 -9.32922944e-12]
 [ 9.92971581e-12  1.19730924e-11 -5.88763468e-11 -8.68747081e-12
  -9.32922944e-12  7.28652237e-18]]

Eigenvalues:
[7.84059847e-20 1.74122611e-04 2.89099853e-04 3.95047397e-04
 5.31372091e-04 7.09760353e-04]

Original propagated covariance:
[[ 4.02426742e-03  8.89394350e-04  1.66336639e-04  1.09052158e-03
   2.27199728e-04 -1.90023917e-10]
 [ 8.89394350e-04  5.41332876e-03  1.67925240e-04  2.29024187e-04
   1.57692825e-03 -1.13550909e-10]
 [ 1.66336639e-04  1.67925240